# Lab 3: Daten einlesen, aufbereiten, explorieren, visualisieren

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# Datenordner finden: Notebook liegt in labs/ oder loesungen/, die Daten in data/
DATA = next(p for p in [Path("data"), Path("../data"), Path("../../data")] if p.exists())
print("Datenordner:", DATA)

Dieses Lab gehört zu **Teil 3: Daten einlesen, aufbereiten, explorieren und visualisieren**. Sie bearbeiten es in sechs kurzen Blöcken zwischen den Folien.

**Lernziele**

- Sie lesen CSV-Dateien mit den passenden Parametern ein und erkennen typische Einlesefehler (Trennzeichen, PLZ mit führender Null, Datum als Text)
- Sie holen Daten mit `pd.read_sql` aus einer Datenbank und kennen den Stolperstein mit Spaltennamen, die SQL-Schlüsselwörter sind
- Sie prüfen einen Datensatz systematisch: Größe, Typen, Kennzahlen, fehlende Werte als Anzahl und als Anteil
- Sie behandeln fehlende Werte, Duplikate, Datentypen und Ausreißer nachvollziehbar und ohne `inplace=True`
- Sie bauen Diagramme mit matplotlib (`fig, ax`) und seaborn: Histogramm, Countplot, Boxplot, Scatterplot, Heatmap
- Sie belegen eine Aussage mit einem eigenen, sauber beschrifteten Diagramm

**So arbeiten Sie:** Jede Aufgabe hat eine eigene Codezelle mit einem Gerüst. Ersetzen Sie `...` durch Ihren Code. Unter jeder Aufgabe steht ein Kontrollergebnis, mit dem Sie sich selbst prüfen.

Leitdatensatz ist `titanic.csv`. Für die Einlesefallen kommen `winequality-red.csv` und die synthetischen Versichertendaten `versicherte.csv` dazu.

In [ ]:
import tempfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 30)

# temporäres Verzeichnis für alle Dateien, die dieses Lab schreibt (CSV, PNG)
TMP = Path(tempfile.mkdtemp())

# Leitdatensatz
df = pd.read_csv(DATA / "titanic.csv")
print("Titanic:", df.shape)
df.head()

## Block 1: Einlesen

1. Lesen Sie `winequality-red.csv` einmal ohne Parameter und einmal mit `sep=";"` ein. Vergleichen Sie `shape` und `dtypes`. **Erwartet: ohne `sep` (1599, 1), also alles in einer Spalte; mit `sep=";"` (1599, 12), davon 11 Spalten `float64` und `quality` als `int64`**
2. Lesen Sie `versicherte.csv` einmal ohne Parameter und einmal mit `parse_dates=["geburtsdatum"]` und `dtype={"plz": str}` ein. Wie viele Postleitzahlen haben ohne `dtype` nur vier Stellen? **Erwartet: ohne Parameter ist `plz` eine Zahl (`int64`) und 361 Postleitzahlen haben nur vier Stellen (die führende Null fehlt); mit Parametern haben alle 5025 Postleitzahlen fünf Stellen und `geburtsdatum` ist ein Datum (`datetime64`)**
3. Schreiben Sie die kleine Tabelle `klein` mit Semikolon und Dezimalkomma in das temporäre Verzeichnis `TMP` und lesen Sie sie korrekt wieder ein. **Erwartet: falsch eingelesen (ohne Parameter) eine einzige Spalte, richtig eingelesen (3, 3), `beitrag_eur` ist `float64` mit Mittelwert 305.45, die PLZ 04109 behält ihre Null**
4. Dieselben Versichertendaten liegen auch als SQLite-Datenbank `versicherte.db` vor (Tabelle `versicherte`). Holen Sie mit `pd.read_sql` je Bundesland die Anzahl und die mittleren Leistungsausgaben der Versicherten mit mindestens 10 Arztbesuchen im Jahr, absteigend nach Anzahl. **Erwartet: 5025 Zeilen in der Tabelle; das Ergebnis hat 16 Zeilen, oben steht Berlin mit 221 Versicherten und 5526.41 Euro**
5. Stolperstein Spaltenname: Berechnen Sie per SQL das mittlere Alter je Geschlecht. Führen Sie die Abfrage zuerst mit `AVG(alter)` aus und lesen Sie die Fehlermeldung. Schreiben Sie den Spaltennamen danach in doppelte Anführungszeichen. **Erwartet: ohne Anführungszeichen `near "alter": syntax error`, weil `ALTER` ein SQL-Schlüsselwort ist (`ALTER TABLE`); mit `"alter"` 2529 Frauen mit 52.3 Jahren und 2496 Männer mit 50.3 Jahren im Mittel**

In [ ]:
# Aufgabe 1: Wein-Datei ohne und mit Trennzeichen einlesen
# Tipp: pd.read_csv(DATA / "winequality-red.csv", sep=";")
wein_falsch = ...
wein = ...
# print(wein_falsch.shape, wein.shape)
# wein.dtypes

In [ ]:
# Aufgabe 2: Versichertendaten ohne und mit Parametern einlesen
# Tipp: astype(str).str.len() liefert die Zahl der Stellen je PLZ
vers_falsch = ...
vers = ...
# print(vers_falsch["plz"].dtype, (vers_falsch["plz"].astype(str).str.len() == 4).sum())
# print(vers["plz"].str.len().value_counts().to_dict(), vers["geburtsdatum"].dtype)

In [ ]:
# Aufgabe 3: kleine CSV mit Semikolon und Dezimalkomma schreiben und wieder einlesen
klein = pd.DataFrame({
    "stadt": ["Leipzig", "Berlin", "Hamburg"],
    "plz": ["04109", "10115", "20095"],
    "beitrag_eur": [312.5, 298.75, 305.1],
})
pfad = TMP / "klein.csv"
# Tipp: klein.to_csv(pfad, index=False, sep=";", decimal=",")
...
# Tipp: beim Einlesen dieselben Angaben, dazu dtype={"plz": str}
zurueck_falsch = ...
zurueck = ...
# print(zurueck_falsch.shape, zurueck.shape, zurueck["beitrag_eur"].dtype, zurueck["beitrag_eur"].mean().round(2))
zurueck

In [ ]:
# Aufgabe 4: Daten per SQL aus einer Datenbank lesen
import sqlite3

con = sqlite3.connect(DATA / "versicherte.db")     # Verbindung öffnen
# Tipp: pd.read_sql("SELECT COUNT(*) AS n FROM versicherte", con)
anzahl = ...
sql = """
    SELECT bundesland, COUNT(*) AS anzahl,
           AVG(leistungsausgaben_eur) AS ausgaben_mittel
    FROM versicherte
    -- hier fehlen WHERE, GROUP BY und ORDER BY
"""
# je_land = pd.read_sql(sql, con)
# print(anzahl, je_land.shape)
# je_land.head(3).round(2)

In [ ]:
# Aufgabe 5: Stolperstein: die Spalte heißt wie ein SQL-Schlüsselwort
# Schritt 1: Kommentarzeichen entfernen, ausführen, Fehlermeldung lesen
# pd.read_sql("SELECT geschlecht, AVG(alter) AS alter_mittel FROM versicherte GROUP BY geschlecht", con)

# Schritt 2: den Spaltennamen in doppelte Anführungszeichen setzen: "alter"
# Tipp: den SQL-Text dann in einfache Anführungszeichen schreiben
alter_je_geschlecht = ...
alter_je_geschlecht

## Block 2: Überblick und fehlende Werte

Ab hier arbeiten Sie mit dem Titanic-Datensatz in `df`.

1. Verschaffen Sie sich den Überblick: `shape`, `info()`, `describe()`. Zählen Sie dann die fehlenden Werte je Spalte, als Anzahl und als Anteil in Prozent, absteigend sortiert. **Erwartet: (891, 12); Cabin 687 (77.10 %), Age 177 (19.87 %), Embarked 2 (0.22 %)**
2. Wie viele Zeilen bleiben nach `dropna()` ohne Parameter, wie viele nach `dropna(subset=["Age"])`? **Erwartet: 183 und 714**
3. Füllen Sie `Age` einmal mit dem Mittelwert, einmal mit dem Median (jeweils in eine neue Series, `df` bleibt unverändert) und vergleichen Sie `describe()` vorher und nachher. **Erwartet: Mittelwert 29.70, Median 28.0; die Standardabweichung sinkt von 14.53 auf 13.00 (Mittelwert) und 13.02 (Median), weil 177 Werte auf denselben Punkt gesetzt werden**
4. Bauen Sie die bereinigte Tabelle `feat_df`: Spalten auswählen, Flag-Spalte `Age_missing` anlegen, `Age` mit dem Median füllen, `Embarked` mit dem Modus füllen, aus `Cabin` die Flag-Spalte `Cabin_bekannt` machen und `Cabin` danach entfernen. **Erwartet: 891 Zeilen, 10 Spalten, keine fehlenden Werte mehr; `Age_missing` 177 mal 1, `Cabin_bekannt` 204 mal 1, Modus von `Embarked` ist S**

In [ ]:
# Aufgabe 1: Überblick und fehlende Werte
# Tipp: isna().sum() zählt, isna().mean() * 100 ergibt den Anteil in Prozent
print(df.shape)
# df.info()
# df.describe().round(2)
fehlend_anzahl = ...
fehlend_prozent = ...
# pd.DataFrame({"anzahl": fehlend_anzahl, "prozent": fehlend_prozent}).head(4)

In [ ]:
# Aufgabe 2: Wie viele Zeilen bleiben nach dropna?
# Tipp: len(df.dropna(...))
zeilen_alle = ...
zeilen_age = ...
# print(zeilen_alle, zeilen_age)

In [ ]:
# Aufgabe 3: Age mit Mittelwert und mit Median füllen, describe vergleichen
# Tipp: df["Age"].fillna(wert) gibt eine neue Series zurück, df bleibt unverändert
age_mittel = ...
age_median = ...
# pd.DataFrame({"original": df["Age"].describe(),
#               "mit_mittelwert": age_mittel.describe(),
#               "mit_median": age_median.describe()}).round(2)

In [ ]:
# Aufgabe 4: bereinigte Tabelle feat_df bauen
spalten = ["Survived", "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked", "Cabin"]
feat_df = df[spalten].copy()

# 1. Markieren: fehlte das Alter? (erst markieren, dann füllen)
feat_df["Age_missing"] = ...
# 2. Age mit dem Median füllen (Ergebnis wieder zuweisen, kein inplace)
feat_df["Age"] = ...
# 3. Embarked mit dem Modus füllen. Tipp: feat_df["Embarked"].mode()[0]
feat_df["Embarked"] = ...
# 4. Cabin: nur behalten, ob eine Kabine bekannt ist. Tipp: notna().astype(int)
feat_df["Cabin_bekannt"] = ...
# feat_df = feat_df.drop(columns=["Cabin"])

# print(feat_df.shape, feat_df.isna().sum().sum())

## Block 3: Duplikate, Typen, Ausreißer

Die nächste Zelle stellt `feat_df` sicher bereit. Wenn Sie Block 2 abgeschlossen haben, ändert sich nichts.

1. Duplikate: Prüfen Sie Titanic und die Versichertendaten (`vers` aus Block 1) auf doppelte Zeilen. Entfernen Sie die Duplikate der Versichertendaten und prüfen Sie danach, ob `versicherten_nr` eindeutig ist. **Erwartet: Titanic 0 Duplikate; Versichertendaten 25 Duplikate, danach 5000 Zeilen und `is_unique` ist `True`**
2. Datentypen: Wandeln Sie in `feat_df` die Spalten `Sex` und `Embarked` in den Typ `category` um und geben Sie die Kategorien aus. **Erwartet: `['female', 'male']` und `['C', 'Q', 'S']`**
3. Boolean-Masken: Wählen Sie (a) alle Männer, die mehr als 20 für ihr Ticket bezahlt haben, und (b) die fünf ältesten Passagiere der ersten Klasse. **Erwartet: (a) 204 Zeilen, (b) der älteste ist 80 Jahre alt**
4. Ausreißer: Berechnen Sie für `Fare` die IQR-Grenzen und zählen Sie die markierten Tickets. Begrenzen Sie `Fare` danach mit `clip` auf das 1-%- und das 99-%-Quantil (neue Spalte `Fare_clip`) und vergleichen Sie `describe()`. **Erwartet: Obergrenze 65.63, 116 markierte Tickets; nach dem Clipping sinkt das Maximum von 512.33 auf 249.01, der Median bleibt 14.45**

In [ ]:
# Aufholzelle: feat_df sicher anlegen (identisch mit Block 2, Aufgabe 4)
feat_df = df[["Survived", "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked", "Cabin"]].copy()
feat_df["Age_missing"] = feat_df["Age"].isna().astype(int)
feat_df["Age"] = feat_df["Age"].fillna(feat_df["Age"].median())
feat_df["Embarked"] = feat_df["Embarked"].fillna(feat_df["Embarked"].mode()[0])
feat_df["Cabin_bekannt"] = feat_df["Cabin"].notna().astype(int)
feat_df = feat_df.drop(columns=["Cabin"])

# Versichertendaten mit den richtigen Parametern (identisch mit Block 1, Aufgabe 2)
vers = pd.read_csv(DATA / "versicherte.csv", parse_dates=["geburtsdatum"], dtype={"plz": str})
print(feat_df.shape, vers.shape)

In [ ]:
# Aufgabe 1: Duplikate finden und entfernen
# Tipp: duplicated().sum(), drop_duplicates(), is_unique
dup_titanic = ...
dup_vers = ...
vers_sauber = ...
# print(dup_titanic, dup_vers, len(vers_sauber), vers_sauber["versicherten_nr"].is_unique)

In [ ]:
# Aufgabe 2: Sex und Embarked als category
# Tipp: astype("category"), Ergebnis wieder zuweisen; Kategorien über .cat.categories
feat_df["Sex"] = ...
feat_df["Embarked"] = ...
# print(list(feat_df["Sex"].cat.categories), list(feat_df["Embarked"].cat.categories))
# feat_df.dtypes

In [ ]:
# Aufgabe 3: Boolean-Masken
# (a) alle Männer mit Fare über 20. Tipp: zwei Bedingungen in Klammern, verknüpft mit &
maenner_teuer = ...
# (b) die fünf ältesten Passagiere der ersten Klasse. Tipp: Maske, dann sort_values und head
aelteste_erste = ...
# print(len(maenner_teuer))
aelteste_erste

In [ ]:
# Aufgabe 4: IQR-Regel für Fare, danach Clipping
q1 = ...
q3 = ...
iqr = ...
untergrenze = ...
obergrenze = ...
# maske = (df["Fare"] < untergrenze) | (df["Fare"] > obergrenze)
# print(round(untergrenze, 2), round(obergrenze, 2), maske.sum())

# Clipping auf das 1-%- und 99-%-Quantil. Tipp: df["Fare"].quantile([0.01, 0.99]), dann clip(lower=..., upper=...)
df["Fare_clip"] = ...
# df[["Fare", "Fare_clip"]].describe().round(2)

## Block 4: matplotlib-Grundgerüst

Jedes Diagramm beginnt mit `fig, ax = plt.subplots()`. Gezeichnet und beschriftet wird über `ax`.

1. Zeichnen Sie ein Histogramm von `Fare` (30 Klassen) mit Titel und deutscher Achsenbeschriftung und speichern Sie es als PNG in `TMP`. **Erwartet: stark rechtsschiefe Verteilung; der erste Balken (0 bis etwa 17) enthält 496 Tickets; die Datei `fare_histogramm.png` existiert**
2. Zeichnen Sie zwei Histogramme nebeneinander: links `Age` der Überlebenden, rechts `Age` der nicht Überlebenden. **Erwartet: 290 Überlebende und 424 nicht Überlebende mit Altersangabe; bei den Überlebenden fällt der Gipfel bei den Kleinkindern auf**
3. Zeichnen Sie `Age` nebeneinander mit `bins=10` und `bins=50`. Beschreiben Sie in einem Satz, was sich am Eindruck ändert. **Erwartet: Mit 10 Klassen wirkt die Verteilung glatt, der höchste Balken enthält 177 Passagiere; mit 50 Klassen sehen Sie Einzelheiten wie den Gipfel bei den Kleinkindern, aber auch mehr zufälliges Zickzack**

In [ ]:
# Aufgabe 1: Histogramm von Fare, beschriftet und gespeichert
fig, ax = plt.subplots(figsize=(8, 5))
# Ihr Code: ax.hist(..., bins=30), ax.set_title(...), ax.set_xlabel(...), ax.set_ylabel(...)
...
fig.tight_layout()
# fig.savefig(TMP / "fare_histogramm.png", dpi=150)     # savefig vor plt.show()
plt.show()

In [ ]:
# Aufgabe 2: Age der Überlebenden und der nicht Überlebenden nebeneinander
# Tipp: ax.hist kommt mit NaN nicht zurecht, deshalb dropna()
alter_ja = ...      # Age der Überlebenden (Survived == 1), ohne fehlende Werte
alter_nein = ...    # Age der nicht Überlebenden

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
# Ihr Code: axes[0].hist(...), axes[1].hist(...), Titel und Achsenbeschriftungen
...
fig.tight_layout()
plt.show()

In [ ]:
# Aufgabe 3: dieselben Daten mit 10 und mit 50 Klassen
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
# Ihr Code: links bins=10, rechts bins=50, jeweils mit Titel und Achsenbeschriftung
...
fig.tight_layout()
plt.show()
# Ihr Satz: Was ändert sich am Eindruck?
#

## Block 5: seaborn-Grundformen

Das Muster aller seaborn-Aufrufe: `sns.<diagramm>(data=df, x=..., y=..., hue=..., ax=ax)`. Die nächste Zelle legt die Hilfsspalte `Überlebt` mit den Werten „nein" und „ja" an, damit die Legende lesbar ist.

1. Histogramm von `Fare` mit `kde=True`. **Erwartet: dieselbe rechtsschiefe Form wie in Block 4, dazu eine geglättete Dichtekurve**
2. Countplot von `Pclass` mit `hue="Überlebt"`. **Erwartet: In der dritten Klasse stehen 372 nicht Überlebende gegen 119 Überlebende, in der ersten Klasse 80 gegen 136**
3. Zwei Boxplots nebeneinander: `Fare` nach `Pclass` und `Age` nach `Pclass`. Schreiben Sie einen Satz: Was sehen Sie? **Erwartet: Median des Ticketpreises 60.29 / 14.25 / 8.05 (Klasse 1 / 2 / 3), Median des Alters 37 / 29 / 24**
4. Scatterplot `Age` gegen `Fare` mit `hue="Überlebt"` und `alpha=0.7`. **Erwartet: 714 Punkte (Passagiere ohne Altersangabe lässt seaborn weg), Punktwolke ohne erkennbare Richtung, bei den teuren Tickets überwiegen die Überlebenden**

In [ ]:
# Hilfsspalte für lesbare Legenden
df["Überlebt"] = df["Survived"].map({0: "nein", 1: "ja"})
df["Überlebt"].value_counts()

In [ ]:
# Aufgabe 1: Histogramm von Fare mit Dichtekurve
fig, ax = plt.subplots(figsize=(8, 5))
# Ihr Code: sns.histplot(data=df, x=..., bins=30, kde=True, ax=ax), Titel und Achsen
...
plt.show()

In [ ]:
# Aufgabe 2: Countplot der Passagierklasse, aufgeteilt nach Überlebt
fig, ax = plt.subplots(figsize=(7, 4.5))
# Ihr Code: sns.countplot(data=df, x=..., hue=..., ax=ax), Titel und Achsen
...
plt.show()
# Kontrolle in Zahlen: pd.crosstab(df["Pclass"], df["Überlebt"])

In [ ]:
# Aufgabe 3: Boxplots Fare nach Pclass und Age nach Pclass nebeneinander
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
# Ihr Code: sns.boxplot(data=df, x="Pclass", y=..., ax=axes[0]) und dasselbe für axes[1]
...
fig.tight_layout()
plt.show()
# Kontrolle in Zahlen: df.groupby("Pclass")[["Fare", "Age"]].median()
# Ihr Satz: Was sehen Sie?
#

In [ ]:
# Aufgabe 4: Scatterplot Age gegen Fare, gefärbt nach Überlebt
fig, ax = plt.subplots(figsize=(8, 5))
# Ihr Code: sns.scatterplot(data=df, x=..., y=..., hue=..., alpha=0.7, ax=ax), Titel und Achsen
...
plt.show()

## Block 6: Korrelation, Heatmap und Ihr eigenes Diagramm

1. Berechnen Sie die Korrelation aller Zahlenspalten von `df` mit `Survived`. **Erwartet: Pclass -0.34, Fare 0.26, PassengerId nahe 0 (-0.01)**
2. Erzeugen Sie aus der bereinigten Tabelle `feat_df` eine Heatmap der Korrelationsmatrix, inklusive `Age_missing` und `Cabin_bekannt`. **Erwartet: Matrix mit 8 Zeilen und 8 Spalten; Pclass und Fare -0.55, Cabin_bekannt und Pclass -0.73, Cabin_bekannt und Survived 0.32, Age_missing und Survived -0.09**
3. Abschlussaufgabe: Bauen Sie ein eigenes Diagramm, das die Aussage „In der dritten Klasse überlebten weniger Passagiere" belegt. Mit Titel und deutschen Achsen, als PNG in `TMP` gespeichert. **Erwartet: Überlebensrate 63 % in der ersten, 47 % in der zweiten, 24 % in der dritten Klasse**

In [ ]:
# Aufgabe 1: Korrelation mit Survived
# Tipp: df.corr(numeric_only=True) liefert die ganze Matrix, ["Survived"] eine Spalte daraus
korr_survived = ...
korr_survived

In [ ]:
# Aufgabe 2: Heatmap der Korrelationsmatrix von feat_df
corr = ...          # Tipp: feat_df.corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(8, 6))
# Ihr Code: sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=ax), Titel
...
plt.show()

In [ ]:
# Aufgabe 3 (Abschluss): Diagramm zur Aussage "In der dritten Klasse überlebten weniger Passagiere"
# Überlegen Sie zuerst: Anzahl oder Rate? Welcher Diagrammtyp passt zur Frage?
# Tipp: Der Mittelwert der 0/1-Spalte Survived je Klasse ist die Überlebensrate.
fig, ax = plt.subplots(figsize=(7, 4.5))
...
# fig.savefig(TMP / "ueberleben_nach_klasse.png", dpi=150)
plt.show()

## Zusatzaufgaben

1. Berechnen Sie die IQR-Grenzen für `Age` und zeigen Sie die markierten Passagiere an. Sind das Fehler? **Erwartet: Grenzen -6.69 und 64.81, 11 markierte Passagiere, alle 65 Jahre oder älter: plausible Werte, keine Fehler**
2. Versichertendaten: Sehen Sie sich in `vers_sauber` die zehn höchsten `leistungsausgaben_eur` an. Fehler oder Hochkostenfälle? **Erwartet: höchster Wert 82170.55 Euro, der zehnthöchste 51238.92 Euro; der Median aller Versicherten liegt bei 1605.33 Euro. Das sind Hochkostenfälle, keine Fehler: Niemand würde sie löschen**
3. Dasselbe Vorgehen wie bei Titanic auf die Versichertendaten: Verteilung der `leistungsausgaben_eur` als Histogramm und Boxplot je `altersgruppe` (Erwachsen unter 65, Senior ab 65), nebeneinander. **Erwartet: stark rechtsschiefe Verteilung; Median 1282.14 Euro bei den Erwachsenen, 3663.43 Euro bei den Senioren**

In [ ]:
# Zusatz 1: IQR-Regel für Age
# Tipp: wie bei Fare; quantile überspringt fehlende Werte
q1_age = ...
q3_age = ...
# ...
markiert = ...
markiert

In [ ]:
# Zusatz 2: die zehn höchsten Leistungsausgaben
# Tipp: nlargest(10, "leistungsausgaben_eur")
top10 = ...
top10

In [ ]:
# Zusatz 3: Versichertendaten visualisieren
# Tipp: altersgruppe mit pd.cut(vers_sauber["alter"], bins=[0, 65, 120], labels=[...], right=False)
vers_plot = vers.drop_duplicates()        # ohne die 25 doppelten Zeilen
vers_plot["altersgruppe"] = ...
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
# Ihr Code: links sns.histplot, rechts sns.boxplot je altersgruppe
...
fig.tight_layout()
plt.show()

## Was Sie mitnehmen

- Nach jedem Einlesen prüfen Sie `shape`, `head()` und `info()`. Falsches Trennzeichen, PLZ ohne führende Null und Datum als Text fallen dort sofort auf.
- Fehlende Werte erst zählen, dann je Spalte entscheiden: löschen, füllen (Median, Modus) oder markieren. Erst markieren, dann füllen, und das Ergebnis immer wieder zuweisen. Die IQR-Regel liefert Kandidaten, keine Urteile.
- Jedes Diagramm folgt dem Gerüst `fig, ax = plt.subplots()`, seaborn zeichnet mit `data=`, `x=`, `y=`, `hue=`, `ax=` hinein. Eine Aussage je Diagramm, Achsen beschriftet, Balken ab 0.